# ARENA3 2단계 — S2C 판정·확인

**S2A·S2B 노트북이 둘 다 끝난 뒤** 실행합니다(새 런타임 하나).
1. 개발 판정: W4·B′·G1·S2A_pre·S2A·S2B_pre·S2B를 1단계와 같은 심판·같은 84상황으로 M4와 비교합니다.
2. 확인: 개발 판정을 **통과한 최종 학습기**(S2A·S2B 중 주 지표가 큰 하나)만 새 배치 C1·C2 84상황에서 M4·P0와 다시 잽니다.
   - 통과한 학습기가 없으면 SUMO를 돌리지 않고 끝납니다.
   - 승자는 처음 한 번 고정되어, 다시 실행해도 바뀌지 않습니다.
   - 사전학습만 한 모델(_pre)은 강화학습 성과가 아니므로 승자 후보가 아닙니다.
3. 시간: 승자가 있으면 기준선 252회 + 승자 252회 = 504회(약 2시간). 끊기면 **모두 실행**하면 끝난 판은 건너뜁니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_ARENA3_S2.zip"
A3 = DRIVE / "arena3"
S2 = A3 / "stage2"
DATA = A3 / "bprime_data"
WORKSPACE = Path("/content/FutureTraffic_ARENA3_S2")
assert ARCHIVE.is_file(), f"묶음이 없습니다: {ARCHIVE}"
assert (DATA / "experiment.json").exists(), f"B′ 교사 자료가 없습니다: {DATA}"
CPUS = os.cpu_count() or 1
print("CPU 개수:", CPUS, "| 결과 폴더:", S2)

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "ARENA3_S2_V1", "ARENA3 2단계 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("arena3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| 2단계 설정 {package_manifest['stage2_config_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_arena3*.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 개발 판정(84상황, M4 대비). 아직 평가가 안 끝난 후보가 있으면 멈춥니다.
BASE = A3 / "referee" / "baseline"
CANDIDATES = [f"W4={BASE}", f"BPRIME={A3 / 'referee' / 'bprime'}", f"G1={A3 / 'referee' / 'G1'}"]
CANDIDATES += [f"{n}={S2 / 'eval' / n}" for n in ("S2A_pre", "S2A", "S2B_pre", "S2B")]
judge_args = ["src/arena3_stage2_eval.py", "judge", "--baseline", str(BASE), "--out", str(S2 / "judge")]
for c in CANDIDATES:
    judge_args += ["--candidate", c]
code = colab.run_live(judge_args, WORKSPACE, check=False)
assert code == 0, "아직 평가가 끝나지 않은 후보가 있습니다(위 '아직 평가 중'). S2A·S2B 노트북이 끝난 뒤 다시 실행하세요."
display(Markdown((S2 / "judge" / "summary.md").read_text()))

In [ ]:
# 2) 확인(C1·C2 84상황): 통과한 최종 학습기 중 주 지표가 가장 큰 하나만 다시 잽니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
BASE = A3 / "referee" / "baseline"  # 판정 칸(5)을 건너뛰고 이 칸만 실행해도 되도록 여기서도 정한다
# 지도 캐시가 아직 없으면 작업자 여러 개가 동시에 만들다 서로 부딪힐 수 있어, 한 번만 미리 만든다.
# (이미 있으면 아무것도 하지 않는다. 소스 파일은 건드리지 않는다.)
colab.run_live(["-c", "import sys; sys.path.insert(0, 'src'); from focused_eval import CACHE, dqn_template_network; "
                "print('지도 캐시:', dqn_template_network(CACHE).name)"], WORKSPACE)
colab.run_live(["src/arena3_stage2_eval.py", "confirm", "--baseline", str(BASE),
                "--candidate", f"S2A={S2 / 'eval' / 'S2A'}", "--candidate", f"S2B={S2 / 'eval' / 'S2B'}",
                "--train", f"S2A={S2 / 'S2A_train'}", "--train", f"S2B={S2 / 'S2B_train'}",
                "--out", str(S2 / "confirm"), "--work", "/content/s2_confirm", "--workers", str(CPUS)], WORKSPACE)
result = colab.read_json(S2 / "confirm" / "confirmation.json")
print("확인 결과:", result["status"], "| 승자:", result["selection"]["winner"])
if (S2 / "confirm" / "summary.md").exists():
    display(Markdown((S2 / "confirm" / "summary.md").read_text()))